# 02 — Feature Extraction

Reads `data_raw.csv` (output of notebook 01) and extracts **23 features** across 5 groups:

| Group | Features | What it captures |
|-------|----------|------------------|
| Readability | Flesch RE, FK Grade, Gunning Fog, Dale-Chall, SMOG | Linguistic complexity |
| Linguistic | Avg sentence length, vocab richness, avg word length, long word ratio, sentence count, word count, content word ratio | Structural patterns |
| Sensitivity | Violence, profanity, adult, drug scores | Mature content markers |
| Sentiment | Positive, negative, neutral, compound (VADER) | Emotional tone |
| Style | Dialogue ratio, exclamation ratio, question ratio | Writing register |

Output: `data_features.csv`

## Imports

In [1]:
import pandas as pd
import numpy as np
import nltk
import textstat
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt',                    quiet=True)
nltk.download('punkt_tab',               quiet=True)
nltk.download('stopwords',               quiet=True)
nltk.download('vader_lexicon',           quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✅ Libraries loaded!")

✅ Libraries loaded!


## Load Dataset

In [2]:
df = pd.read_csv('data_raw.csv')

# Normalise label format: accept both '+4' and '4' style labels
df['age_group'] = df['age_group'].astype(str).str.strip()
if not df['age_group'].str.startswith('+').any():
    df['age_group'] = '+' + df['age_group']

print(f"✅ Loaded {len(df)} samples")
print(df['age_group'].value_counts().sort_index())
df.head(3)

✅ Loaded 8000 samples
age_group
+10    2000
+12    2000
+18    2000
+4     2000
Name: count, dtype: int64


,text,age_group
0,"Once upon a time in a bustling city, there was...",+10
1,In a small town lived a curious little boy nam...,+10
2,Once upon a time there was a boy called Tom. H...,+4


## 1 — Readability Features

Five established readability scores — each measures linguistic complexity from a different angle:

- **Flesch Reading Ease** — higher = easier (~90 for children's text, ~20 for academic)
- **Flesch-Kincaid Grade** — maps directly to US school grade level
- **Gunning Fog** — years of education needed to understand the text
- **Dale-Chall** — compares vocabulary against 3,000 familiar words
- **SMOG Index** — grade estimate based on polysyllabic word count

In [3]:
def get_readability_features(text):
    try:
        return {
            'flesch_reading_ease':  textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
            'gunning_fog':          textstat.gunning_fog(text),
            'dale_chall':           textstat.dale_chall_readability_score(text),
            'smog_index':           textstat.smog_index(text),
        }
    except:
        return {k: np.nan for k in ['flesch_reading_ease', 'flesch_kincaid_grade',
                                    'gunning_fog', 'dale_chall', 'smog_index']}

print("Extracting readability features...")
readability = df['text'].apply(get_readability_features).apply(pd.Series)
print("✅ Done!")

print("\nMean readability per class:")
print(pd.concat([df[['age_group']].reset_index(drop=True), readability], axis=1)
      .groupby('age_group')[['flesch_reading_ease','flesch_kincaid_grade']]
      .mean().round(2).sort_index())
readability.head(3)

Extracting readability features...
✅ Done!

Mean readability per class:
           flesch_reading_ease  flesch_kincaid_grade
age_group                                           
+10                      61.83                  8.57
+12                      56.27                  9.89
+18                      51.62                 11.44
+4                       88.11                  3.73


,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,dale_chall,smog_index
0,60.358999,8.643317,10.344182,9.315259,11.362044
1,63.280891,7.728724,8.522747,9.452064,9.725611
2,94.655625,3.463780,5.694048,6.208239,6.182691


## 2 — Linguistic Features

Seven structural and vocabulary features that capture *how* language is used:

- **avg_sentence_length** — longer sentences = more mature writing
- **vocab_richness** — unique words / total words; higher = more varied vocabulary
- **avg_word_length** — longer words tend to appear in advanced texts
- **long_word_ratio** — proportion of words over 6 characters
- **num_sentences / num_words** — raw structural counts
- **content_word_ratio** — proportion of non-stopwords; higher = more information-dense

In [4]:
def get_linguistic_features(text):
    try:
        sentences = sent_tokenize(text)
        words = word_tokenize(text.lower())
        words = [w for w in words if w.isalpha()]
        stop_words = set(stopwords.words('english'))
        content_words = [w for w in words if w not in stop_words]

        return {
            'avg_sentence_length': np.mean([len(word_tokenize(s)) for s in sentences]) if sentences else 0,
            'vocab_richness':      len(set(words)) / len(words) if words else 0,
            'avg_word_length':     np.mean([len(w) for w in words]) if words else 0,
            'long_word_ratio':     len([w for w in words if len(w) > 6]) / len(words) if words else 0,
            'num_sentences':       len(sentences),
            'num_words':           len(words),
            'content_word_ratio':  len(content_words) / len(words) if words else 0,
        }
    except:
        return {k: np.nan for k in ['avg_sentence_length', 'vocab_richness', 'avg_word_length',
                                    'long_word_ratio', 'num_sentences', 'num_words', 'content_word_ratio']}

print("Extracting linguistic features...")
linguistic = df['text'].apply(get_linguistic_features).apply(pd.Series)
print("✅ Done!")
linguistic.head(3)

Extracting linguistic features...
✅ Done!


,avg_sentence_length,vocab_richness,avg_word_length,long_word_ratio,num_sentences,num_words,content_word_ratio
0,15.533333,0.753769,4.929648,0.266332,15.0,199.0,0.628141
1,15.866667,0.743719,5.020101,0.266332,15.0,199.0,0.608040
2,15.928571,0.520833,3.703125,0.052083,14.0,192.0,0.510417


## 3 — Sensitivity Features

Four keyword-based scores (per 1,000 words) measuring the presence of mature content.
Scores are normalised by text length so short and long texts are comparable.

In [5]:
VIOLENCE_WORDS = set([
    "kill","killed","killing","killer","kills",
    "murder","murdered","murderer","murders",
    "stab","stabbed","stabbing","shoot","shooting","shoots",
    "gun","guns","pistol","rifle","firearm","bullet","bullets",
    "fight","fought","fighting","punch","punched","punching","kick","kicked",
    "attack","attacked","attacking","assault","assaulted","assaults",
    "beat","beaten","beating","slap","slapped",
    "blood","bloody","bleed","bled","bleeding","bloodshed",
    "wound","wounded","wounding","injury","injured","injuries",
    "death","died","dying","corpse","skull",
    "war","battle","combat","soldier","army","troops","weapon","weapons",
    "bomb","explosion","explode","exploded","destroy","destroyed","blast",
    "torture","torment","brutality","brutal","gore","gory","gruesome",
    "slaughter","massacre","carnage","violence","violent","violently",
    "threaten","threat","threatened","choke","strangle","suffocate",
    "knife","sword","axe","dagger","spear",
    "abuse","abused","abuser","predator","victim","hostage",
    "horror","terror","terrifying","terrified","nightmare",
    "scar","scarred","agony","suffer","suffering","suffered",
    "execute","execution","hang","hanged","behead","beheaded",
    "riot","destruction","wreck","wreckage",
])

PROFANITY_WORDS = set([
    "damn","damned","dammit","damnit","crap","crappy",
    "bastard","bitch","bitches","piss","pissed","prick",
    "shit","shitty","bullshit","bollocks","bugger",
    "fuck","fucked","fucker","fucking","fucks","motherfucker",
    "cock","asshole","arsehole","wanker","tosser",
    "whore","slut","skank",
])

ADULT_WORDS = set([
    "sex","sexual","sexually","sexuality","sexualise","sexualize",
    "naked","nude","nudity","undressed","undress",
    "erotic","erotica","erection","arousal","aroused","arouse",
    "orgasm","climax","intercourse","penetration",
    "breast","breasts","nipple","nipples","cleavage",
    "penis","vagina","genitals","genitalia","groin",
    "porn","pornography","pornographic","obscene","obscenity",
    "rape","raped","rapist","molest","molested","molestation",
    "prostitute","prostitution","brothel",
    "seduce","seduced","seduction","seductive","seducing",
    "lust","lustful","lusty",
    "affair","adultery","infidelity",
    "sensual","sensuality","sensuous",
    "condom","contraception","abortion",
])

DRUG_WORDS = set([
    "alcohol","alcoholic","alcoholism","drunk","drunken","drunkenness","intoxicated",
    "beer","wine","whiskey","whisky","vodka","tequila","rum","gin","liquor","booze",
    "hangover",
    "cigarette","cigarettes","smoking","smoker",
    "tobacco","cigar","cigars","vaping","vape","nicotine",
    "cocaine","crack","heroin","marijuana","cannabis","weed","pot","hash","hashish",
    "meth","methamphetamine","amphetamine","ecstasy","mdma","lsd","mushroom",
    "overdose","addict","addiction","addicted","junkie",
    "snort","snorted","snorting","stoned",
    "hallucinate","hallucination","tripping","blunt","joint","bong",
    "opioid","opioids","fentanyl","xanax","valium","sedative",
])

def get_sensitivity_features(text):
    """Score sensitivity keywords per 1000 words (normalised for text length)."""
    try:
        words = word_tokenize(text.lower())
        alpha_words = [w for w in words if w.isalpha()]
        total  = max(len(alpha_words), 1)
        per_k  = 1000 / total
        return {
            'violence_score':  round(sum(1 for w in alpha_words if w in VIOLENCE_WORDS)  * per_k, 4),
            'profanity_score': round(sum(1 for w in alpha_words if w in PROFANITY_WORDS) * per_k, 4),
            'adult_score':     round(sum(1 for w in alpha_words if w in ADULT_WORDS)     * per_k, 4),
            'drug_score':      round(sum(1 for w in alpha_words if w in DRUG_WORDS)      * per_k, 4),
        }
    except:
        return {k: np.nan for k in ['violence_score', 'profanity_score', 'adult_score', 'drug_score']}

print("Extracting sensitivity features...")
sensitivity = df['text'].apply(get_sensitivity_features).apply(pd.Series)
print("✅ Done!")
print("\nMean sensitivity scores per class:")
temp = pd.concat([df[['age_group']].reset_index(drop=True), sensitivity], axis=1)
print(temp.groupby('age_group')[['violence_score','profanity_score','adult_score','drug_score']]
      .mean().round(4).sort_index())
sensitivity.head(3)

Extracting sensitivity features...
✅ Done!

Mean sensitivity scores per class:
           violence_score  profanity_score  adult_score  drug_score
age_group                                                          
+10                1.9608           0.0030       0.0281      0.1708
+12                5.2370           0.0404       0.2577      0.2507
+18               10.6026           0.0519       1.2964      0.4364
+4                 0.9279           0.0116       0.0073      0.2216


,violence_score,profanity_score,adult_score,drug_score
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0


## 4 — Sentiment Features

NLTK VADER scores the emotional tone of each text. Children's texts tend to be more positive and neutral; adult fiction often carries higher negative and compound variation.

In [6]:
sia = SentimentIntensityAnalyzer()

def get_sentiment_features(text):
    try:
        scores = sia.polarity_scores(text)
        return {
            'sentiment_positive': scores['pos'],
            'sentiment_negative': scores['neg'],
            'sentiment_neutral':  scores['neu'],
            'sentiment_compound': scores['compound'],
        }
    except:
        return {k: np.nan for k in ['sentiment_positive', 'sentiment_negative',
                                    'sentiment_neutral',  'sentiment_compound']}

print("Extracting sentiment features...")
sentiment = df['text'].apply(get_sentiment_features).apply(pd.Series)
print("✅ Done!")

print("\nMean sentiment per class:")
temp = pd.concat([df[['age_group']].reset_index(drop=True), sentiment], axis=1)
print(temp.groupby('age_group')[['sentiment_positive','sentiment_negative','sentiment_compound']]
      .mean().round(3).sort_index())
sentiment.head(3)

Extracting sentiment features...
✅ Done!

Mean sentiment per class:
           sentiment_positive  sentiment_negative  sentiment_compound
age_group                                                            
+10                     0.166               0.047               0.733
+12                     0.138               0.084               0.387
+18                     0.117               0.112               0.008
+4                      0.186               0.049               0.753


,sentiment_positive,sentiment_negative,sentiment_neutral,sentiment_compound
0,0.156,0.023,0.821,0.9648
1,0.147,0.077,0.776,0.9319
2,0.254,0.025,0.721,0.9940


## 5 — Style Features

Three features that capture writing register — signals readability scores miss entirely:

- **dialogue_ratio** — proportion of text inside quotes; children's/YA fiction use heavy dialogue
- **exclamation_ratio** — exclamation marks as a share of all sentence-ending punctuation
- **question_ratio** — questions as a share of all sentence-ending punctuation

In [7]:
def get_style_features(text):
    quoted        = re.findall(r'"[^"]*"', text)
    quoted_chars  = sum(len(q) for q in quoted)
    sentences     = max(text.count('.') + text.count('!') + text.count('?'), 1)
    return {
        'dialogue_ratio':    quoted_chars / max(len(text), 1),
        'exclamation_ratio': text.count('!') / sentences,
        'question_ratio':    text.count('?') / sentences,
    }

style = df['text'].apply(get_style_features).apply(pd.Series)

print("=== Style feature means per class ===")
temp = pd.concat([df[['age_group']].reset_index(drop=True), style], axis=1)
print(temp.groupby('age_group')[['dialogue_ratio','exclamation_ratio','question_ratio']]
      .mean().round(4).sort_index())
style.head(3)

=== Style feature means per class ===
           dialogue_ratio  exclamation_ratio  question_ratio
age_group                                                   
+10                0.1359             0.1316          0.0758
+12                0.0441             0.0338          0.0533
+18                0.0163             0.0065          0.0362
+4                 0.0763             0.0916          0.0232


,dialogue_ratio,exclamation_ratio,question_ratio
0,0.243221,0.357143,0.071429
1,0.069936,0.071429,0.142857
2,0.259615,0.000000,0.000000


## Combine & Save

Joins all five feature groups into a single DataFrame alongside the age group label.  
Rows with any NaN values are dropped. Output saved as `data_features.csv`.

In [8]:
features_df = pd.concat([
    df[['age_group']].reset_index(drop=True),
    readability.reset_index(drop=True),
    linguistic.reset_index(drop=True),
    sensitivity.reset_index(drop=True),
    sentiment.reset_index(drop=True),
    style.reset_index(drop=True),
], axis=1)

features_df = features_df.dropna().reset_index(drop=True)

feature_cols = [c for c in features_df.columns if c != 'age_group']
features_df.to_csv('data_features.csv', index=False)

print(f"✅ Saved data_features.csv — Shape: {features_df.shape}")
print(f"Features ({len(feature_cols)} total): {feature_cols}")
print(f"\nClass distribution after feature extraction:")
print(features_df['age_group'].value_counts().sort_index())

✅ Saved data_features.csv — Shape: (8000, 24)
Features (23 total): ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog', 'dale_chall', 'smog_index', 'avg_sentence_length', 'vocab_richness', 'avg_word_length', 'long_word_ratio', 'num_sentences', 'num_words', 'content_word_ratio', 'violence_score', 'profanity_score', 'adult_score', 'drug_score', 'sentiment_positive', 'sentiment_negative', 'sentiment_neutral', 'sentiment_compound', 'dialogue_ratio', 'exclamation_ratio', 'question_ratio']

Class distribution after feature extraction:
age_group
+10    2000
+12    2000
+18    2000
+4     2000
Name: count, dtype: int64


## Feature Means per Class (Summary Table)

In [9]:
summary = features_df.groupby('age_group')[feature_cols].mean().round(3).sort_index().T
summary.columns.name = None
print("Mean feature values per age group:")
summary

Mean feature values per age group:


,+10,+12,+18,+4
flesch_reading_ease,61.830,56.270,51.623,88.112
flesch_kincaid_grade,8.573,9.885,11.439,3.734
gunning_fog,10.267,11.947,13.729,5.431
dale_chall,9.715,9.980,10.530,6.950
smog_index,10.843,11.868,12.874,6.763
avg_sentence_length,20.508,24.131,31.429,13.369
vocab_richness,0.724,0.714,0.700,0.544
avg_word_length,4.779,4.757,4.706,3.965
long_word_ratio,0.226,0.235,0.233,0.104
num_sentences,11.634,9.378,6.063,14.721
